# Milestone 5 🚩

**Q1)** Dataset Splitting Strategy Task: You are building a lazy-loading "recipe" generator for the 10 genres.

Instead of loading audio into RAM, you will create a list of dictionaries. Assume you create exactly 100 recipe dictionaries per genre. Pass this combined list of recipes into sklearn.model_selection.train_test_split using test_size=0.2, shuffle=True, and random_state=42.

Question: Exactly how many recipe items will be allocated to your validation set (val_recipes) list?

**Ans** Total dataset : `10 (genre) * 100 (per genre recipe dicts) = 1000`  
Test set = `0.2 * 1000 = 200`

**Q2)** On-the-Fly Mixing Dimensions Task: In your training loop, you will load audio stems on the fly. 

Assume you use librosa.load(sr=16000, duration=10) to load 4 instrument stems and 1 noise file. You pad or truncate all 5 arrays to be exactly 160,000 samples long. 

You sum the 4 stems together to create a mix array, and then add the noise array scaled by an intensity weight of 0.2. 


Question: Before this final mix array is passed to the feature extractor, what must its exact 1D NumPy shape be? 

**Ans** It must be a 1d numpy array, so shape is (160000,)

**Q3)** Hugging Face Feature Extractor Shape Task: 


Initialize Hugging Face's AutoFeatureExtractor using the `MIT/ast-finetuned-audioset-10-10-0.4593` checkpoint. 

To test your pipeline, create a dummy audio mix of exactly 160,000 ones using 
mix = np.ones(160000). 
Pass this dummy array into the extractor with sampling_rate=16000 and return_tensors="pt". 
Finally, extract the input_values from the resulting dictionary and apply .squeeze(0) to remove the default batch dimension.  

Question: What is the exact shape of the resulting PyTorch tensor? (This is the spectrogram matrix the AST model will process). In this format: [a,b] or (a,b)

In [1]:
import numpy as np
import torch
from transformers import AutoFeatureExtractor

feature_extractor = AutoFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)
mix = np.ones(160000)
inputs = feature_extractor(
    mix,
    sampling_rate=16000,
    return_tensors="pt"
)
print(inputs["input_values"].squeeze(0).shape)

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

torch.Size([1024, 128])


**Q4)** Model Architecture Initialization Task: 
Initialize the ASTForAudioClassification.from_pretrained model using the "MIT/ast-finetuned-audioset-10-10-0.4593" checkpoint. 

Because we are classifying 10 genres instead of the original AudioSet classes, you must explicitly pass num_labels=10 and ignore_mismatched_sizes=True to replace the classification head. 

Question: Write a quick loop using 

sum(p.numel() for p in model.parameters() if p.requires_grad) 

to calculate the number of trainable parameters in this newly configured model. What is the exact integer value?

In [2]:
from transformers import ASTForAudioClassification

# modified classification head
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
)

sum(p.numel() for p in model.parameters() if p.requires_grad)

2026-03-20 12:46:07.769676: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774010768.081355      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774010768.178056      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774010768.998512      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774010768.998574      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774010768.998578      55 computation_placer.cc:177] computation placer alr

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


86196490

**Q5)** Inference Normalization Math Task: In your test inference loop, right before feature extraction, the audio is normalized to prevent clipping using this exact formula: 

y = y / (np.max(np.abs(y)) + 1e-9). 

To test this, create a NumPy array: 
y_test = np.array([-0.85, 0.40, 0.20, -0.10]). 

Apply the normalization formula to it. 

Question: What is the resulting value at index 0 of the normalized array, rounded to 3 decimal places?

In [3]:
y_test = np.array([-0.85, 0.40, 0.20, -0.10])

y_normalized = y_test / (np.max(np.abs(y_test)) + 1e-9)

print(y_normalized)
print(round(y_normalized[0], 3))

[-1.          0.47058823  0.23529412 -0.11764706]
-1.0
